# Tendon vibration — P03 vs P04

**Only two participants have vibration recordings.** P05's 24-07 session is polarity x lidocaine;
its log does not mention vibration at all. So this is a two-participant comparison, not three.

## The thing that decides how this must be read

The vibrator was on a **different muscle and a different side** in each participant:

| | vibrator on | arm | conditions recorded |
|---|---|---|---|
| **P03** | wrist **extensor** | **left** | baseline, applied-but-off, ON (+ a 5 mA-step repeat for 30 Hz) |
| **P04** | **flexor** tendon | **right** | baseline, OFF, ON flexors, ON **extensors** (a check, vibrator moved) |

So P03's "VIB ON" and P04's "VIB ON extensors" are **not the same condition**. Comparing them
directly would compare left-extensor vibration against right-extensor vibration in different
people.

**What is comparable is the CHANGE at the vibrated site**, measured against that participant's own
baseline. Every bar below is a % of the same participant's baseline at the same intensity;
100 % = unchanged. The vibrated muscle is marked with `*`.

## One more caveat, from the logs

P03's "baseline" block was recorded at 16:31 with the **vibrator already attached** (applied
~15:45, switched off), and ~1.5 h into the session. P04's baseline is at 14:22, before the
vibrator was involved at all. So P03's baseline is closer to P04's *VIB OFF* than to P04's
baseline.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, thresholds_for, threshold_source,
                       vibration_table, fig_vibration, fig_bars,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       check_thresholds, mt_stamp, warn_if_stale,
                       fig_threshold_grid,
                       fig_detection_row)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
D03 = "tSCS_CHUV_data/15-07-2026/P03tscsHealthy/"
D04 = "tSCS_CHUV_data/16-07-2026/P04tscsHealthy/"

RUN = {   # (subject, condition) -> csv.  Set PROTOCOL to "burst" or "arcex".
 "burst": {
   ("P03", "Baseline"):        D03 + "Burst_autosave_20260715_164431_737ms.csv",
   ("P03", "VIB off"):         D03 + "Burst_autosave_20260715_165831_999ms.csv",
   ("P03", "VIB ON"):          D03 + "Burst_autosave_20260715_170629_569ms.csv",
   ("P04", "Baseline"):        D04 + "Burst_autosave_20260716_142311_232ms.csv",
   ("P04", "VIB off"):         D04 + "Burst_autosave_20260716_144116_893ms.csv",
   ("P04", "VIB ON"):          D04 + "Burst_autosave_20260716_145100_665ms.csv",   # flexors
   ("P04", "VIB ON extensors"):D04 + "Burst_autosave_20260716_152821_805ms.csv",
 },
 "arcex": {
   ("P03", "Baseline"):        D03 + "Modulated_autosave_20260715_164630_593ms.csv",
   ("P03", "VIB off"):         D03 + "Modulated_autosave_20260715_170032_049ms.csv",
   ("P03", "VIB ON"):          D03 + "Modulated_autosave_20260715_171433_605ms.csv",
   ("P04", "Baseline"):        D04 + "Modulated_autosave_20260716_142536_785ms.csv",
   ("P04", "VIB off"):         D04 + "Modulated_autosave_20260716_144439_044ms.csv",
   ("P04", "VIB ON"):          D04 + "Modulated_autosave_20260716_145845_958ms.csv",  # flexors
   ("P04", "VIB ON extensors"):D04 + "Modulated_autosave_20260716_153556_059ms.csv",
 },
}
PROTOCOL   = "burst"          # <-- "burst" or "arcex"
SUBJECTS   = ["P03", "P04"]
CONDITIONS = ["Baseline", "VIB off", "VIB ON", "VIB ON extensors"]
VIBRATED   = {"P03": "Ext. digitorum (L)",    # the muscle the vibrator sat on
              "P04": "Flex. digitorum (R)"}
SITE       = {"P03": "left wrist extensor",   # ...in words, for the panel titles
              "P04": "right flexor tendon"}
# the same four muscles as the baseline comparison
MUSCLES = ["Flex. digitorum (L)", "Flex. digitorum (R)", "Ext. digitorum (L)",
           "Flex. carpi rad. (R)"]

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = RUN[PROTOCOL]
for (s, cond), f in runs.items():
    meta = load_run(f)[0]
    print(f"{s}  {cond:18s} {f.split('/')[-1][-22:]}  {[m['amp_ma'] for m in meta][0]}-"
          f"{[m['amp_ma'] for m in meta][-1]} mA")

## 2 · Check the thresholds you chose — before anything else

**Read straight from `results/<session>/mt_*.csv`**, not from any `MT` built later, so this is
what you actually picked and nothing can substitute a different value.

One row per recording, one column per muscle: every intensity stacked, **your threshold in red**,
the mA printed in the corner. The traces below it should be flat; the red one should be the first
with a clear response. `not picked` means that muscle has no threshold saved for that recording.

Wrong? Fix it with the `repick()` cells at the bottom, press Save, and re-run this cell and the
one after it.

In [ ]:
# every recording of this comparison, so the picks can be compared between conditions too
CHECK = {f"{s} · {cond}": runs[(s, cond)] for s in SUBJECTS for cond in CONDITIONS
         if (s, cond) in runs}

picked = fig_threshold_grid(CHECK, MUSCLES,
                            title=f"the thresholds saved for each recording — {PROTOCOL}")

## 2 · Motor thresholds, from each participant's BASELINE

Every condition of a participant is measured at the intensity its **baseline** puts the threshold
at - the manipulation must not be allowed to move the intensity as well, or the comparison is of
two things at once.

In [ ]:
# hand-picked thresholds win wherever they exist; detection fills in the rest. Pick them in
# notebooks/motor_thresholds/ and re-run this cell - the source is printed below.
MT = {s: thresholds_for(runs[(s, "Baseline")], MUSCLES, consecutive=2, **KW)
      for s in SUBJECTS}
for s in SUBJECTS:
    print(f"{s}  ({threshold_source(runs[(s, 'Baseline')])})  "
          + ", ".join(f"{m.split(' (')[0][:9]}{m[-3:]} {v:g}" for m, v in sorted(MT[s].items())))

MT_STAMP = mt_stamp(runs)      # so the figures can tell you if a pick is saved after this

## 3 · Every condition as a change from that participant's baseline

100 % = unchanged. `*` marks the muscle the vibrator was actually on - that bar is the effect;
the others are whether the effect was specific to the vibrated site or general.

In [ ]:
# only the BASELINE recordings set the thresholds these figures use
warn_if_stale(runs, MT_STAMP, used=[(s, "Baseline") for s in SUBJECTS])

tab = vibration_table(runs, MT, MUSCLES, CONDITIONS, base="Baseline", **KW)
fig_vibration(tab, SUBJECTS, MUSCLES, CONDITIONS, vibrated=VIBRATED, site=SITE,
              title=f"Tendon vibration - {PROTOCOL}",
              save=None)

### The numbers, so they can be checked

In [ ]:
print(f"{'subject':8s}{'muscle':22s}{'mA':>5s}" + "".join(f"{c[:15]:>17s}" for c in CONDITIONS))
for s in SUBJECTS:
    for m in MUSCLES:
        row = [tab.get((s, c_, m), {}).get("pct", float("nan")) for c_ in CONDITIONS]
        if not any(v == v for v in row):
            continue
        amp = next((tab[(s, c_, m)]["amp"] for c_ in CONDITIONS if (s, c_, m) in tab), None)
        star = " *" if VIBRATED.get(s) == m else "  "
        print(f"{s:8s}{m + star:22s}{amp:5g}"
              + "".join(f"{(f'{v:.0f} %' if v == v else '-'):>17s}" for v in row))

## 4 · Before vs during vibration, one figure per muscle

The same layout as the baseline comparison: **1st pulse** in mV, then the **2nd pulse** and the
**mean of pulses 2-10** as % of that condition's own 1st pulse, with the **EMG underneath** at the
intensity the bars are measured from.

Here the compared series are the **conditions**, not the protocols: grey = baseline, light =
vibrator applied but off, red = vibration ON. Participants are side by side, each at its own
muscle thresholds from its own baseline recording.

Read the vibrated muscle (`VIBRATED` above) as the effect, and the others as how much the
recording drifted between blocks - if a muscle far from the vibrator moves as much as the vibrated
one, the change is drift, not vibration.

In [ ]:
# ======== settings ========
SHOW_MUSCLES = MUSCLES      # the same four as the baseline comparison
BARS_COND    = ["Baseline", "VIB off", "VIB ON"]     # which conditions to put side by side
COND_COLOUR  = {"Baseline": "#9A9A9A", "VIB off": "#C6C6C6", "VIB ON": "#C0392B",
                "VIB ON extensors": "#2E6F95"}
# ==========================

# fig_bars compares any set of series; here they are conditions, so every condition of a
# participant is measured at the thresholds its BASELINE defines
warn_if_stale(runs, MT_STAMP, used=[(s, "Baseline") for s in SUBJECTS])
MT_cond = {(s, cond): MT[s] for s in SUBJECTS for cond in BARS_COND}

for m in SHOW_MUSCLES:
    fig_bars(runs, MT_cond, m, SUBJECTS, protocols=tuple(BARS_COND), steps=0,
             colours=COND_COLOUR, names={c: c for c in BARS_COND},
             min_snr_ratio=1.5, rest_from=2, with_p1=True, traces=True,
             title=m, save=None, **KW)

## 5 · How the peaks are detected — every muscle, one row per recording

Each panel is one muscle at **its own threshold**: the 10 pulses re-aligned on their own onset,
with the max (v) and min (^) that the peak-to-peak is made of.

| | |
|---|---|
| **green** | the response window — `RESP_START_MS` after the pulse to `GUARD_MS` before the next |
| **red bands** | the only places a max/min can be taken — `ANCHOR_WIN_MS` either side of the train's own average latency. **A peak outside these is skipped however clear it looks** |
| markers stacked | the same deflection on every pulse |
| red ring | flagged: on a window border, or at a different latency from the rest |
| corner | how many of the 10 pulses were flagged |

### The knobs, and which way loosens

`TRY` below overrides the analysis settings **for this figure only**, so you can see the effect of
loosening something without changing any result. When you find settings you want, copy them into
`KW` in the config cell and re-run the notebook.

| setting | now | loosen by | what it does |
|---|---|---|---|
| `anchor_win_ms` | 3.0 | **raising** it, or `anchor=None` | how far from the train's average latency a peak may be. The usual reason a visible peak is missed |
| `resp_start_ms` | 8.0 | **lowering** it | how long after the pulse the window opens. Too low and it takes the artifact |
| `min_snr` | 1.2 | **lowering** it | how far above baseline noise a train must be to count at all |
| `max_edge_frac` | 0.5 | **raising** it, or `None` | how many pulses may have their peak on a window border before the train is called artifact |

In [ ]:
# ======== try different detection settings, for this figure only ========
ROW = ("P04", "Baseline")            # which recording to inspect
TRY = dict(KW)                       # start from the analysis settings...
# TRY["anchor_win_ms"] = 6.0         # ...and loosen one at a time
# TRY["anchor"] = None               # no anchoring: take the biggest max/min in the window
# TRY["resp_start_ms"] = 6.0
# TRY["min_snr"] = None
# TRY["max_edge_frac"] = None
# =======================================================================

fig_detection_row(runs[ROW], MT[ROW[0]], MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                  title=f"{ROW[0]} · {ROW[1]} — peak detection at each muscle's own threshold",
                  **TRY)

### Change a threshold here, without leaving the notebook

One cell per recording. Move the slider to the trace you want, press **Save**, then **re-run the
motor-threshold cell above** — every figure below follows.

It writes the same `results/<session>/mt_<recording>.csv` as
`notebooks/motor_thresholds/`, so the two never disagree, and re-opening a cell reloads what you
saved so you can correct it again.

> **Which recording actually moves these figures.** Every condition is measured at the intensity
> that participant's **Baseline** recording defines, so re-picking `('P03', 'VIB ON')` or any other
> non-baseline block changes **nothing here** - that is the design, not a bug. Only re-picking a
> **Baseline** does. The figure cells say so if you re-pick something else.


In [ ]:
def repick(key):
    """Open the picker for one recording of this comparison, with a Save button.

    The green line is the DETECTOR's answer for this recording - not the baseline, and not
    whatever is already saved (that is the orange trace)."""
    csv = RUN[key] if isinstance(RUN, dict) and key in RUN else runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    print("   the analysis uses this participant's BASELINE thresholds: "
          + ", ".join(f"{m.split(' (')[0][:9]}{m[-3:]} {v:g}" for m, v in sorted(MT[key[0]].items())
                      if m in MUSCLES))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('P03', 'Baseline'))

In [ ]:
repick(('P03', 'VIB off'))

In [ ]:
repick(('P03', 'VIB ON'))

In [ ]:
repick(('P04', 'Baseline'))

In [ ]:
repick(('P04', 'VIB off'))

In [ ]:
repick(('P04', 'VIB ON'))

In [ ]:
repick(('P04', 'VIB ON extensors'))